# `latincy_vocab` — a standalone spaCy/LatinCy vocabulary component

`latincy_vocab` consumes an already-parsed spaCy `Doc` and sets `doc._.vocab_list`
(a `VocabList`) — a deduplicated, reading-order vocabulary list. It **consumes upstream
token annotations** (lemma, POS, and `token._.gloss` from latincy-lexicon's
`whitakers_words`) rather than loading any gloss/lexicon files itself.

- Proper names are excluded (they route to a separate NER/NEL channel).
- Standalone enclitics (`-que`) are dropped.
- Glosses are optional: with no upstream gloss pipe, the list is still produced (lexicon-free).

In [ ]:
import spacy
import vocabbuilder  # importing registers the `latincy_vocab` factory
from vocabbuilder import VocabList

PASSAGE = (
    "Q. Mucius augur multa narrare de C. Laelio socero suo memoriter et iucunde "
    "solebat nec dubitare illum in omni sermone appellare sapientem; ego autem a "
    "patre ita eram deductus ad Scaeuolam sumpta uirili toga, ut, quoad possem et "
    "liceret, a senis latere numquam discederem."
)

## 1. Lexicon-free — `nlp.add_pipe("latincy_vocab")`

Just add the component to any LatinCy pipeline. No gloss data needed.

In [ ]:
nlp = spacy.load("la_core_web_lg")
nlp.add_pipe("latincy_vocab")

doc = nlp(PASSAGE)
vocab = doc._.vocab_list
assert isinstance(vocab, VocabList)

print(f"{len(vocab)} entries (deduped, no PROPN, no enclitics)\n")
for e in vocab.by_first_occurrence():
    print(f"{e.display_lemma:<14} {e.pos:<6} x{e.frequency}")

## 2. With glosses — consume `whitakers_words` upstream

Add latincy-lexicon's `whitakers_words` **before** `latincy_vocab`; the component
picks up each token's `token._.gloss`. (Citation-form headwords will arrive the same
way once the latincy-lexicon refactor lands — no change needed here.)

In [ ]:
from pathlib import Path
import latincy_lexicon

# Resolve the lexicon data shipped with the installed (editable) latincy-lexicon.
LEX = Path(latincy_lexicon.__file__).resolve().parents[2] / "data" / "json"

nlp_g = spacy.load("la_core_web_lg")
nlp_g.add_pipe(
    "whitakers_words",
    config={"lexicon_path": str(LEX / "lexicon.json"), "analyzer_path": str(LEX / "analyzer.json")},
)
nlp_g.add_pipe("latincy_vocab")
print("pipeline:", nlp_g.pipe_names)

glossed = nlp_g(PASSAGE)._.vocab_list
for e in glossed.by_first_occurrence():
    gloss = e.glosses[0] if e.glosses else "—"
    print(f"{e.display_lemma:<14} {e.pos:<6} {gloss}")

## 3. Views & export

`VocabList` offers `by_frequency()`, `by_alpha()`, `by_first_occurrence()`,
`filter_pos()`, `filter_min_frequency()`, plus `to_markdown()` / `to_json()` / `to_dicts()`.

In [ ]:
# Nouns and verbs only, as a Markdown glossary
content = glossed.filter_pos({"NOUN", "VERB"}).by_alpha()
print(content.to_markdown())

In [ ]:
# JSON for downstream consumers (e.g. the reader's substrate)
print(glossed.by_frequency().to_json()[:600], "...")